# Architecture: Encoder-Decoder Transformer (bidirectional encoder) — v2

Legal LM capstone, Phase 1. ~62M params, trained from scratch with the shared byte-level BPE tokenizer.

**Changes vs model_01**

*Bugs fixed*
- Full-run config restored (no more 50-step debug settings); every run gets its own `output_dir`.
- Checkpoints store `encoder_causal` + an `arch_name`. Loading **refuses** a config without `encoder_causal`, because the causal/bidirectional switch is a mask flag, not a weight, so `load_state_dict` can never detect the mismatch.
- Resume from `last.pt` (model + optimizer + scaler + step + best val + planned total steps). Batches depend only on the step, so resumption is exact.
- Data prep: held-out (validation) documents can no longer leak into training shards after the val token cap is reached.

*Training*
- BERT-style MLM micro-batches on the encoder (1 in every `mlm_every`), with 15% masking and the 80/10/10 replacement rule. Gives every encoder position a direct loss, instead of only gradients through cross-attention.
- Document-boundary-aware window sampling. A window is rejected if a BOS/EOS falls in the decoder span or in the last `boundary_guard` encoder tokens.
- Lengths matched to the test conditions: `decoder_len=256` (128 words ≈ 182 tokens at 1.42 tokens/word), and `encoder_len` set from the measured prompt length.
- `total_steps` auto-sized from measured throughput, so the cosine schedule actually finishes inside the time budget.
- No weight decay on norms and embeddings. Residual init scaled per stack (1/√(2L) for the encoder, 1/√(3L) for the decoder, which has three residual branches).
- Mixed precision: bf16 on Ampere+ GPUs, fp16 + GradScaler on T4/P100.

*Inference*
- Cross-attention K/V computed once per encoder pass (previously recomputed at every decoding step in every layer).
- Generation continues until the text has more than 128 words (adaptive chunks), so outputs are never short or empty.
- Repetition penalty and n-gram blocking see the whole generated history, across chunks. Both are vectorised on the GPU.
- Beam search (KV caches reordered per beam), alongside greedy and temperature / top-k / top-p sampling.
- Optional word-boundary constraint: the first generated token can't glue onto the prompt's last word.
- Pseudo-test set from held-out documents, with local ROUGE-L / BLEU-4 / repetition metrics and a decoding sweep.

*How to run (Kaggle, GPU)*
1. Cells 1–9 set things up. Cell 9 runs the self-tests; they must all pass.
2. Cell 10 measures the test prompt lengths. Cell 11 trains, with an auto-sized schedule and a hard stop at `session_hours`, and resumes if re-run.
3. Cell 12 runs the decoding sweep on the pseudo-test set. Copy the best setting into `FINAL_DECODE` in Cell 13, which writes and validates `submission.zip`.
4. Inference-only session: run Cells 1–10 (skip training by setting `RUN_TRAIN=False` in Cell 11), point `CKPT_PATH` at your best.pt dataset, then run Cells 12–13.

*Verification*
- Self-tests (run them first): KV-cache equivalence, beam(1)==greedy, padding invariance, bidirectionality, n-gram ban, checkpoint guard, loader boundary stats.
- Submission validator: exact ID set, exactly 128 words, no U+FFFD, no prompt leakage, zip root.

In [1]:
# ===================== Cell 1: setup =====================
import os, sys, json, math, time, random, shutil, zipfile, hashlib, contextlib
from pathlib import Path
from dataclasses import dataclass, asdict, fields
from typing import Optional, Dict, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- HF login (Kaggle secret "HF"; skipped elsewhere) ----
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF login skipped:", e)

SESSION_START = time.time()   # wall-clock start of this session (training hard-stops relative to it)

# ---- Paths ----
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

_TOKENIZER_CANDIDATES = [
    Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large"),
    Path("/kaggle/input/capstone-tokenizer-large"),
    Path("/kaggle/working/capstone_tokenizer"),
]
TOKENIZER_DIR = next((p for p in _TOKENIZER_CANDIDATES if p.exists()), None)
if TOKENIZER_DIR is None:
    raise FileNotFoundError("Tokenizer artifacts not found; add the tokenizer dataset.")

# ---- Device / precision ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    # T4 (sm_75) and P100 (sm_60) have no native bf16 -> fp16 + GradScaler.
    _major = torch.cuda.get_device_capability()[0]
    AMP_DTYPE = torch.bfloat16 if _major >= 8 else torch.float16
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0))
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | amp={USE_AMP} ({AMP_DTYPE}) | grad scaler={USE_SCALER}")

def amp_ctx(enabled: bool = True):
    if device.type == "cuda" and USE_AMP and enabled:
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()

# ---- Reproducibility ----
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---- Tokenizer ----
sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer
tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = tokenizer.vocab_size
BOS_ID, EOS_ID, PAD_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id
_special_to_id = getattr(tokenizer, "special_to_id", {})
MASK_ID = _special_to_id.get("<mask>", 6)
# Ids below this are specials / reserved / sentinels (0..122); byte tokens start at 123.
FIRST_VALID_ID = len(getattr(tokenizer, "special_names", [])) or 123
assert FIRST_VALID_ID == 123, FIRST_VALID_ID

# Tokens that would *continue* a word (first byte alphanumeric or non-ASCII).
# Test prompts end on a word boundary, so the first generated token should not glue onto the last word.
WORD_CONT_MASK = torch.zeros(VOCAB_SIZE, dtype=torch.bool)
_id2b = getattr(tokenizer, "id_to_bytes", {})
for _i in range(FIRST_VALID_ID, VOCAB_SIZE):
    _b = _id2b.get(_i)
    if _b:
        _c = _b[0]
        if _c >= 0x80 or chr(_c).isalnum():
            WORD_CONT_MASK[_i] = True
WORD_CONT_MASK = WORD_CONT_MASK.to(device)
print(f"vocab={VOCAB_SIZE} BOS={BOS_ID} EOS={EOS_ID} MASK={MASK_ID} first_valid={FIRST_VALID_ID} "
      f"word-continuation tokens={int(WORD_CONT_MASK.sum())}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok
GPU: Tesla T4
device=cuda | amp=True (torch.float16) | grad scaler=True
vocab=32000 BOS=2 EOS=3 MASK=6 first_valid=123 word-continuation tokens=10374


## Data preparation
Reuses existing shards if they are present (in `/kaggle/working`, or in `PREBUILT_SHARD_DIR` if you saved them as a dataset). It also saves raw held-out documents to `val_texts.jsonl`; the pseudo-test set is built from them.

In [2]:
# ===================== Cell 2: data preparation =====================
%pip install -q ijson

import ijson
from array import array
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm

HF_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_PATHS = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}
TEST_PATH = "Phase_1_Pretraining/Test/Phase1- Test.json"

PREBUILT_SHARD_DIR = None          # e.g. Path("/kaggle/input/<your-shards-dataset>/shards")
SHARD_DIR = Path(PREBUILT_SHARD_DIR) if PREBUILT_SHARD_DIR else WORK_DIR / "shards"
SHARD_TOKENS  = 50_000_000
VAL_FRACTION  = 0.005
VAL_CAP_TOTAL = 5_000_000
TOKEN_CAPS = {"acts": 20_000_000, "high_courts": 65_000_000,
              "supreme_court": 25_000_000, "tribunals": 40_000_000}

VAL_TEXTS_PATH = WORK_DIR / "val_texts.jsonl"   # raw held-out docs for the pseudo-test set
VAL_TEXTS_PER_SOURCE = 60
VAL_TEXT_MIN_WORDS = 700                         # need >= 512 + 128 words

def clean_text(t):
    return (t or "").replace("\r\n", "\n").strip()

def doc_id_of(doc):
    return str(doc.get("url") or doc.get("id") or doc.get("_idx"))

def is_val(doc_id, salt="phase1"):
    h = int(hashlib.md5((salt + "::" + doc_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000.0 < VAL_FRACTION

def stream_docs(path_in_repo):
    fs = HfFileSystem()
    with fs.open(f"datasets/{HF_REPO}/{path_in_repo}", "rb") as f:
        for i, doc in enumerate(ijson.items(f, "item")):
            if isinstance(doc, dict):
                doc.setdefault("_idx", i)
                yield doc

def shards_ready(d=SHARD_DIR):
    return all((Path(d) / s / "meta.json").exists() for s in SOURCE_PATHS)

def encode_all():
    summary = {}
    vt_f = open(VAL_TEXTS_PATH, "w")
    for source, path_in_repo in SOURCE_PATHS.items():
        out_dir = SHARD_DIR / source
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"\n[{source}] streaming {path_in_repo}")
        meta = {"source": source, "hf_path": path_in_repo, "shards": [], "total_tokens": 0,
                "total_bytes": 0, "num_docs": 0, "val_policy": "strict"}
        shard_ids, val_ids = array("H"), array("H")
        shard_idx, n_vt = 0, 0
        val_cap, cap = VAL_CAP_TOTAL // len(SOURCE_PATHS), TOKEN_CAPS[source]

        def flush():
            nonlocal shard_ids, shard_idx
            if len(shard_ids) == 0:
                return
            arr = np.frombuffer(shard_ids, dtype=np.uint16)
            final = out_dir / f"shard_{shard_idx:04d}.bin"
            tmp = final.with_suffix(".bin.tmp")
            arr.tofile(tmp); tmp.rename(final)
            meta["shards"].append({"name": final.name, "tokens": int(arr.size)})
            meta["total_tokens"] += int(arr.size)
            print(f"  [{source}] {final.name}: {arr.size/1e6:.1f}M tokens")
            shard_ids = array("H"); shard_idx += 1

        pbar = tqdm(desc=f"  {source}", unit="doc")
        try:
            for doc in stream_docs(path_in_repo):
                pbar.update(1)
                text = clean_text(doc.get("text"))
                if not text:
                    continue
                did = doc_id_of(doc)
                if is_val(did):
                    # FIX: held-out docs NEVER go to training, even after the val cap is reached.
                    if len(val_ids) < val_cap:
                        val_ids.append(BOS_ID)
                        val_ids.extend(tokenizer.encode(text, allow_special=False))
                        val_ids.append(EOS_ID)
                    if n_vt < VAL_TEXTS_PER_SOURCE and len(text.split()) >= VAL_TEXT_MIN_WORDS:
                        vt_f.write(json.dumps({"id": did, "source": source, "text": text}) + "\n")
                        n_vt += 1
                    continue
                meta["num_docs"] += 1
                meta["total_bytes"] += len(text.encode("utf-8"))
                shard_ids.append(BOS_ID)
                shard_ids.extend(tokenizer.encode(text, allow_special=False))
                shard_ids.append(EOS_ID)
                if len(shard_ids) >= SHARD_TOKENS:
                    flush()
                if meta["total_tokens"] + len(shard_ids) >= cap:
                    print(f"  [{source}] hit cap {cap/1e6:.0f}M")
                    break
        finally:
            pbar.close()
        flush()
        if len(val_ids) > 0:
            arr = np.frombuffer(val_ids, dtype=np.uint16)
            (out_dir / "val.bin").write_bytes(arr.tobytes())
            meta["val_tokens"] = int(arr.size)
        (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
        summary[source] = meta
        print(f"[{source}] {meta['total_tokens']/1e6:.1f}M tokens, {meta['num_docs']} docs, "
              f"val={meta.get('val_tokens', 0)/1e6:.2f}M, val texts={n_vt}")
    vt_f.close()
    return summary

def collect_val_texts(max_scan_per_source=150_000):
    # Used only when shards already exist but val_texts.jsonl does not. Streams raw docs, keeps held-out ones.
    with open(VAL_TEXTS_PATH, "w") as f:
        for source, path_in_repo in SOURCE_PATHS.items():
            n = 0
            for i, doc in enumerate(stream_docs(path_in_repo)):
                if n >= VAL_TEXTS_PER_SOURCE or i >= max_scan_per_source:
                    break
                text = clean_text(doc.get("text"))
                did = doc_id_of(doc)
                if text and is_val(did) and len(text.split()) >= VAL_TEXT_MIN_WORDS:
                    f.write(json.dumps({"id": did, "source": source, "text": text}) + "\n")
                    n += 1
            print(f"[{source}] collected {n} held-out docs")

if shards_ready():
    print("Shards found at", SHARD_DIR)
    _m = json.loads((SHARD_DIR / "acts" / "meta.json").read_text())
    if _m.get("val_policy") != "strict":
        print("WARNING: these shards were built with the old val policy (held-out docs past the val cap "
              "were trained on). Pseudo-test scores may be slightly optimistic; rebuild if time permits.")
    if not VAL_TEXTS_PATH.exists():
        collect_val_texts()
else:
    encode_all()
    print("Data preparation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.

[acts] streaming Phase_1_Pretraining/Train/train_acts.json


  acts: 0doc [00:00, ?doc/s]

  [acts] hit cap 20M
  [acts] shard_0000.bin: 20.0M tokens
[acts] 20.0M tokens, 2218 docs, val=0.07M, val texts=9

[high_courts] streaming Phase_1_Pretraining/Train/train_high_courts.json


  high_courts: 0doc [00:00, ?doc/s]

  [high_courts] shard_0000.bin: 50.0M tokens
  [high_courts] hit cap 65M
  [high_courts] shard_0001.bin: 15.0M tokens
[high_courts] 65.0M tokens, 112192 docs, val=0.31M, val texts=57

[supreme_court] streaming Phase_1_Pretraining/Train/train_supreme_court.json


  supreme_court: 0doc [00:00, ?doc/s]

  [supreme_court] hit cap 25M
  [supreme_court] shard_0000.bin: 25.0M tokens
[supreme_court] 25.0M tokens, 3632 docs, val=0.12M, val texts=17

[tribunals] streaming Phase_1_Pretraining/Train/train_tribunals.json


  tribunals: 0doc [00:00, ?doc/s]

  [tribunals] hit cap 40M
  [tribunals] shard_0000.bin: 40.0M tokens
[tribunals] 40.0M tokens, 8423 docs, val=0.20M, val texts=30
Data preparation complete.


## Model

In [3]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Data loaders
Two tasks are sampled. **Prefix-LM seq2seq:** the encoder gets `encoder_len` tokens and the decoder predicts the next `decoder_len`. **MLM:** the encoder alone gets a corrupted window. Seq2seq windows are rejected if a document boundary falls in the decoder span or in the last `boundary_guard` encoder tokens.

In [4]:
# ===================== Cell 4: data loaders =====================
class ShardReader:
    def __init__(self, source_dir):
        self.dir = Path(source_dir)
        meta = json.loads((self.dir / "meta.json").read_text())
        self.shards = meta["shards"]
        self.total_tokens = sum(s["tokens"] for s in self.shards)
        self.arrays = [np.memmap(self.dir / s["name"], dtype=np.uint16, mode="r") for s in self.shards]
        w = np.array([s["tokens"] for s in self.shards], dtype=np.float64)
        self.shard_weights = w / w.sum()
        vp = self.dir / "val.bin"
        self.val_arr = np.memmap(vp, dtype=np.uint16, mode="r") if vp.exists() else None

def has_boundary(w, start):
    seg = w[start:]
    return bool(((seg == BOS_ID) | (seg == EOS_ID)).any())

def mlm_corrupt(ids, rng, p=0.15):
    # BERT masking: choose p of the eligible tokens; 80% -> <mask>, 10% -> random token, 10% unchanged.
    ids = ids.copy()
    labels = np.full_like(ids, -100)
    sel = (rng.random(ids.shape) < p) & (ids >= FIRST_VALID_ID)
    if not sel.any():                               # guarantee at least one target per batch
        r, c = np.nonzero(ids >= FIRST_VALID_ID)
        if len(r):
            j = int(rng.integers(len(r))); sel[r[j], c[j]] = True
    labels[sel] = ids[sel]
    u = rng.random(ids.shape)
    to_mask = sel & (u < 0.8)
    to_rand = sel & (u >= 0.8) & (u < 0.9)
    ids[to_mask] = MASK_ID
    ids[to_rand] = rng.integers(FIRST_VALID_ID, VOCAB_SIZE, size=int(to_rand.sum()))
    return ids, labels

class PretrainLoader:
    def __init__(self, sources, mix_weights, encoder_len, decoder_len, micro_batch, seed=0,
                 boundary_guard=64, max_tries=8, mlm_prob=0.15):
        self.sources = sources
        self.keys = list(sources.keys())
        p = np.array([mix_weights[k] for k in self.keys], dtype=np.float64)
        self.p = p / p.sum()
        self.E, self.D, self.B = encoder_len, decoder_len, micro_batch
        self.seed, self.guard, self.max_tries, self.mlm_prob = seed, boundary_guard, max_tries, mlm_prob
        self.stats = {"ok": 0, "fallback": 0}

    def _rng(self, key, task):
        s = (self.seed * 1_000_003 + key * 2_654_435_761 + task * 97_531) & 0xFFFFFFFF
        return np.random.default_rng(s)

    def _sample(self, rng, length, check_from):
        w = None
        for _ in range(self.max_tries):
            reader = self.sources[self.keys[rng.choice(len(self.keys), p=self.p)]]
            arr = reader.arrays[rng.choice(len(reader.arrays), p=reader.shard_weights)]
            if len(arr) <= length:
                continue
            start = int(rng.integers(0, len(arr) - length))
            w = np.asarray(arr[start:start + length], dtype=np.int64)
            if check_from is None or not has_boundary(w, check_from):
                self.stats["ok"] += 1
                return w
        if w is None:
            raise RuntimeError("No shard is longer than the requested window.")
        self.stats["fallback"] += 1
        return w

    def get_s2s_batch(self, key):
        rng = self._rng(key, 0)
        E, D = self.E, self.D
        enc = np.empty((self.B, E), np.int64)
        dec_in = np.empty((self.B, D), np.int64)
        dec_y = np.empty((self.B, D), np.int64)
        for b in range(self.B):
            w = self._sample(rng, E + D, check_from=max(0, E - self.guard))
            enc[b] = w[:E]
            dec_y[b] = w[E:E + D]
            dec_in[b, 0] = BOS_ID
            dec_in[b, 1:] = dec_y[b, :-1]
        return torch.from_numpy(enc), torch.from_numpy(dec_in), torch.from_numpy(dec_y)

    def get_mlm_batch(self, key):
        rng = self._rng(key, 1)
        ids = np.stack([self._sample(rng, self.E, check_from=None) for _ in range(self.B)])
        ids, labels = mlm_corrupt(ids, rng, self.mlm_prob)
        return torch.from_numpy(ids), torch.from_numpy(labels)

    def fallback_rate(self):
        n = self.stats["ok"] + self.stats["fallback"]
        return self.stats["fallback"] / max(1, n)


def _val_windows(reader, length, n_max, check_from):
    out, s, arr = [], 0, reader.val_arr
    stride = max(1, length // 2)
    while s + length <= len(arr) and len(out) < n_max:
        w = np.asarray(arr[s:s + length], dtype=np.int64)
        if check_from is None or not has_boundary(w, check_from):
            out.append(w)
            s += length
        else:
            s += stride
    return out

class FixedValSeq2Seq:
    def __init__(self, sources, encoder_len, decoder_len, windows_per_source=64, batch_size=8, guard=64):
        self.E, self.D, self.bs = encoder_len, decoder_len, batch_size
        self.windows = []
        for name, r in sources.items():
            if r.val_arr is not None:
                self.windows += _val_windows(r, self.E + self.D, windows_per_source, max(0, self.E - guard))
        print(f"FixedValSeq2Seq: {len(self.windows)} windows")
    def __iter__(self):
        for i in range(0, len(self.windows), self.bs):
            ch = np.stack(self.windows[i:i + self.bs])
            enc, dec_y = ch[:, :self.E], ch[:, self.E:self.E + self.D]
            dec_in = np.concatenate([np.full((len(ch), 1), BOS_ID), dec_y[:, :-1]], axis=1)
            yield torch.from_numpy(enc), torch.from_numpy(dec_in), torch.from_numpy(dec_y)

class FixedValMLM:
    def __init__(self, sources, encoder_len, windows_per_source=32, batch_size=8, mlm_prob=0.15):
        self.bs, ws = batch_size, []
        for name, r in sources.items():
            if r.val_arr is not None:
                ws += _val_windows(r, encoder_len, windows_per_source, None)
        rng = np.random.default_rng(12345)          # fixed corruption -> comparable across evals
        self.batches = []
        for i in range(0, len(ws), batch_size):
            ids, lab = mlm_corrupt(np.stack(ws[i:i + batch_size]), rng, mlm_prob)
            self.batches.append((torch.from_numpy(ids), torch.from_numpy(lab)))
        print(f"FixedValMLM: {len(ws)} windows")
    def __iter__(self):
        return iter(self.batches)

## Training
`total_steps=None` sizes the cosine schedule from throughput measured on this GPU, so the schedule completes within `plan_hours`. The run stops at `session_hours`, writes `last.pt`, and resumes when re-run with the same `output_dir`. On a new Kaggle session, attach `last.pt` as a dataset and set `resume_from`.

In [5]:
# ===================== Cell 5: training =====================
@dataclass
class TrainConfig:
    micro_batch: int = 8
    grad_accum: int = 8
    peak_lr: float = 6e-4
    min_lr: float = 6e-5
    warmup_steps: int = 300
    # warmup_steps: int = 10
    total_steps: Optional[int] = None    # None -> auto from measured step time and plan_hours
    plan_hours: float = 7.0              # training time the schedule is planned for
    # plan_hours: float = 0.1              # training time the schedule is planned for
    session_hours: float = 8.0           # hard stop for this session (then resume)
    # session_hours: float = 0.2           # hard stop for this session (then resume)
    bench_steps: int = 20
    # bench_steps: int = 5
    weight_decay: float = 0.1
    betas: Tuple[float, float] = (0.9, 0.95)
    eps: float = 1e-8
    grad_clip: float = 1.0
    mlm_every: int = 4                   # every k-th micro-batch is MLM (0 disables)
    mlm_prob: float = 0.15
    mlm_weight: float = 1.0
    boundary_guard: int = 64
    eval_every: int = 250
    # eval_every: int = 10
    val_windows_per_source: int = 64
    log_every: int = 25
    # log_every: int = 5
    ckpt_every_minutes: float = 20
    # ckpt_every_minutes: float = 0.1
    seed: int = 1337
    output_dir: str = ""
    resume: bool = True
    resume_from: Optional[str] = None    # path to a last.pt from a previous session (e.g. a Kaggle dataset)
    fit_schedule_in_session: bool = True # True: the cosine schedule must also finish before session_hours

def cosine_lr(step, warmup, total, peak, minimum):
    if step < warmup:
        return peak * (step + 1) / warmup
    if total is None:
        return peak
    if step >= total:
        return minimum
    prog = (step - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (peak - minimum) * (1 + math.cos(math.pi * prog))

def make_optimizer(model, tcfg):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if (p.dim() < 2 or "tok_emb" in n or "pos_emb" in n) else decay).append(p)
    print(f"optimizer: {sum(p.numel() for p in decay)/1e6:.2f}M decayed, "
          f"{sum(p.numel() for p in no_decay)/1e6:.2f}M not decayed")
    return torch.optim.AdamW([{"params": decay, "weight_decay": tcfg.weight_decay},
                              {"params": no_decay, "weight_decay": 0.0}],
                             lr=tcfg.peak_lr, betas=tcfg.betas, eps=tcfg.eps)

@torch.no_grad()
def evaluate_s2s(model, val_loader):
    was = model.training; model.eval()
    tot, n = 0.0, 0
    for enc, dec_in, dec_y in val_loader:
        enc, dec_in, dec_y = enc.to(device), dec_in.to(device), dec_y.to(device)
        with amp_ctx():
            logits = model(enc, dec_in)
        tot += F.cross_entropy(logits.float().view(-1, logits.size(-1)), dec_y.view(-1), reduction="sum").item()
        n += dec_y.numel()
    model.train(was)
    return tot / max(1, n)

@torch.no_grad()
def evaluate_mlm(model, val_mlm):
    if val_mlm is None or model.mlm_transform is None:
        return float("nan")
    was = model.training; model.eval()
    tot, n = 0.0, 0
    for ids, lab in val_mlm:
        ids, lab = ids.to(device), lab.to(device)
        sel = lab != -100
        with amp_ctx():
            logits = model.mlm_logits_at(model.encode(ids), sel)
        tot += F.cross_entropy(logits.float(), lab[sel], reduction="sum").item()
        n += int(sel.sum())
    model.train(was)
    return tot / max(1, n)

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

def train_seq2seq(cfg, tcfg, train_loader, val_s2s, val_mlm=None):
    assert tcfg.output_dir, "set a unique output_dir per run"
    out_dir = Path(tcfg.output_dir); out_dir.mkdir(parents=True, exist_ok=True)
    log_path, last_path, best_path = out_dir / "log.jsonl", out_dir / "last.pt", out_dir / "best.pt"
    if tcfg.total_steps is None:
        assert tcfg.warmup_steps > tcfg.bench_steps + 2, "warmup must cover the throughput benchmark"

    model = Seq2SeqLM(cfg).to(device)
    init_weights(model, cfg)
    pc = count_params(model)
    print(f"params: total={pc['total']/1e6:.2f}M | emb={pc['embedding']/1e6:.2f}M | "
          f"non-emb={pc['non_embedding']/1e6:.2f}M")
    assert 50_000_000 <= pc["total"] <= 300_000_000, "param count outside the 50M-300M rule"
    opt = make_optimizer(model, tcfg)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    state = {"step": -1, "best_val": float("inf"), "tokens_seen": 0,
             "total_steps": tcfg.total_steps, "train_seconds": 0.0}

    # ---- resume ----
    if tcfg.resume_from and not last_path.exists():
        shutil.copy(tcfg.resume_from, last_path)
        _b = Path(tcfg.resume_from).with_name("best.pt")
        if _b.exists() and not best_path.exists():
            shutil.copy(_b, best_path)
        print("copied", tcfg.resume_from, "->", last_path)
    if tcfg.resume and last_path.exists():
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        if asdict(cfg_from_dict(ck["config"])) != asdict(cfg):
            raise ValueError("last.pt config differs from the current ModelConfig; use a new output_dir")
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        state.update(ck["state"])
        print(f"RESUMED from step {state['step']} (best val {state['best_val']:.4f}, "
              f"planned total {state['total_steps']})")

    def ckpt_payload(with_opt):
        d = {"model": model.state_dict(), "config": asdict(cfg), "train_config": asdict(tcfg),
             "state": dict(state)}
        if with_opt:
            d["opt"], d["scaler"] = opt.state_dict(), scaler.state_dict()
        return d

    def log(rec):
        with open(log_path, "a") as f:
            f.write(json.dumps(rec) + "\n")

    def do_eval(step):
        vl, vm = evaluate_s2s(model, val_s2s), evaluate_mlm(model, val_mlm)
        print(f"  [eval] step {step} | val s2s {vl:.4f} (ppl {math.exp(vl):.2f}) | val mlm {vm:.4f}")
        log({"step": step, "val_loss": vl, "val_mlm": vm})
        if vl < state["best_val"]:
            state["best_val"], state["val_loss"] = vl, vl
            atomic_save(ckpt_payload(False), best_path)
            print(f"  [best] saved best.pt (val {vl:.4f})")

    model.train()
    ga, V = tcfg.grad_accum, cfg.vocab_size
    t_session = time.time(); last_ckpt = time.time(); t_log = time.time(); tok_log = 0
    bench_t0, bench_from = None, state["step"] + 3
    step = state["step"]
    finished = False

    while True:
        step += 1
        total = state["total_steps"]
        if total is not None and step >= total:
            finished = True
            break
        # ---- auto-size the schedule from measured throughput ----
        if total is None and step == bench_from:
            _sync(); bench_t0 = time.time()
        if total is None and step == bench_from + tcfg.bench_steps:
            _sync()
            sps = (time.time() - bench_t0) / tcfg.bench_steps
            remaining = tcfg.plan_hours * 3600 * 0.93 - state["train_seconds"] - (time.time() - t_session)
            if tcfg.fit_schedule_in_session:
                session_left = tcfg.session_hours * 3600 - (time.time() - SESSION_START)
                remaining = min(remaining, session_left * 0.93)
            total = step + max(100, int(remaining / sps))
            state["total_steps"] = total
            print(f"[schedule] {sps:.2f}s/step -> total_steps={total} "
                  f"(~{(total - step) * sps / 3600:.1f}h of training left)")

        lr = cosine_lr(step, tcfg.warmup_steps, total, tcfg.peak_lr, tcfg.min_lr)
        for g in opt.param_groups:
            g["lr"] = lr
        opt.zero_grad(set_to_none=True)
        s2s_sum = torch.zeros((), device=device); mlm_sum = torch.zeros((), device=device)
        s2s_n = mlm_n = 0
        for micro in range(ga):
            key = step * ga + micro
            is_mlm = (tcfg.mlm_every > 0 and model.mlm_transform is not None
                      and micro % tcfg.mlm_every == tcfg.mlm_every - 1)
            if is_mlm:
                ids, lab = train_loader.get_mlm_batch(key)
                ids, lab = ids.to(device, non_blocking=True), lab.to(device, non_blocking=True)
                sel = lab != -100
                with amp_ctx():
                    logits = model.mlm_logits_at(model.encode(ids), sel)
                loss = F.cross_entropy(logits.float(), lab[sel])
                w = tcfg.mlm_weight
                mlm_sum += loss.detach(); mlm_n += 1
            else:
                enc, dec_in, dec_y = train_loader.get_s2s_batch(key)
                enc, dec_in, dec_y = (enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True),
                                      dec_y.to(device, non_blocking=True))
                with amp_ctx():
                    logits = model(enc, dec_in)
                loss = F.cross_entropy(logits.float().view(-1, V), dec_y.view(-1))
                w = 1.0
                s2s_sum += loss.detach(); s2s_n += 1
                state["tokens_seen"] += dec_y.numel(); tok_log += dec_y.numel()
            scaler.scale(loss * w / ga).backward()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        scaler.step(opt); scaler.update()
        state["step"] = step

        if step % tcfg.log_every == 0:
            s2s_l = (s2s_sum / max(1, s2s_n)).item()
            mlm_l = (mlm_sum / max(1, mlm_n)).item() if mlm_n else float("nan")
            dt = time.time() - t_log
            el = state["train_seconds"] + time.time() - t_session
            print(f"step {step:5d}/{total} | s2s {s2s_l:.4f} | mlm {mlm_l:.4f} | lr {lr:.2e} | "
                  f"gnorm {float(gn):.2f} | {tok_log/max(dt,1e-6)/1e3:.1f}k dec-tok/s | "
                  f"boundary-fallback {train_loader.fallback_rate():.1%} | {el/3600:.2f}h")
            log({"step": step, "loss": s2s_l, "mlm_loss": mlm_l, "lr": lr, "grad_norm": float(gn),
                 "tokens_seen": state["tokens_seen"], "elapsed_s": el})
            t_log, tok_log = time.time(), 0

        if step > 0 and step % tcfg.eval_every == 0:
            do_eval(step)

        if (time.time() - last_ckpt) / 60 >= tcfg.ckpt_every_minutes:
            state["train_seconds"] += time.time() - t_session; t_session = time.time()
            atomic_save(ckpt_payload(True), last_path)
            last_ckpt = time.time()
            print(f"  [ckpt] last.pt @ step {step}")

        if (time.time() - SESSION_START) / 3600 >= tcfg.session_hours:
            print(f"Session time limit reached at step {step}. Re-run with resume to continue.")
            break

    state["train_seconds"] += time.time() - t_session
    if finished:
        do_eval(state["step"])
        atomic_save(ckpt_payload(False), out_dir / "final.pt")
        print("Training finished. best val:", state["best_val"])
    atomic_save(ckpt_payload(True), last_path)
    return model, out_dir, finished

## Generation
Greedy, sampling, and beam search, all with cached cross-attention K/V and a self-attention KV cache. Penalties are vectorised. `hist` holds each row's previously generated tokens (left-padded with -1), so the repetition penalty and n-gram blocking work across chunk boundaries.

In [6]:
# ===================== Cell 6: generation =====================
@dataclass
class DecodeConfig:
    strategy: str = "greedy"          # greedy | sample | beam
    temperature: float = 1.0
    top_k: int = 0
    top_p: float = 1.0
    num_beams: int = 4
    repetition_penalty: float = 1.0
    no_repeat_ngram: int = 0
    history_tokens: int = 512         # how far back the penalties look (generated tokens only)
    force_word_boundary: bool = True  # first token of the continuation must start a new word / punctuation
    def label(self):
        s = self.strategy
        if s == "sample":
            s += f"_T{self.temperature}_k{self.top_k}_p{self.top_p}"
        if s == "beam":
            s += f"{self.num_beams}"
        return f"{s}_rp{self.repetition_penalty}_ng{self.no_repeat_ngram}"

def left_pad(seqs, pad=-1):
    L = max((len(s) for s in seqs), default=0)
    out = torch.full((len(seqs), L), pad, dtype=torch.long)
    for i, s in enumerate(seqs):
        if len(s):
            out[i, L - len(s):] = torch.tensor(s, dtype=torch.long)
    return out.to(device)

def right_pad_batch(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    msk = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s, dtype=torch.long); msk[i, :len(s)] = 1
    if bool(msk.all()):
        msk = None                    # no padding -> no mask -> fastest SDPA path
    return ids.to(device), (msk.to(device) if msk is not None else None)

def build_seen(hist, V):
    seen = torch.zeros(hist.size(0), V, dtype=torch.bool, device=hist.device)
    if hist.numel():
        seen.scatter_(1, hist.clamp(min=0), hist >= 0)   # -1 pads map to id 0 (a special, banned anyway)
    return seen

def apply_penalties(logits, hist, seen, dc, boundary_rows=None):
    logits = logits.clone()
    logits[:, :FIRST_VALID_ID] = float("-inf")           # never emit PAD/UNK/BOS/EOS/reserved/sentinels
    base = logits.clone()
    if dc.repetition_penalty != 1.0:
        rp = dc.repetition_penalty
        pen = torch.where(logits > 0, logits / rp, logits * rp)
        logits = torch.where(seen, pen, logits)
    n = dc.no_repeat_ngram
    if n > 1 and hist.size(1) >= n:
        win = hist.unfold(1, n, 1)                        # [R, L-n+1, n]
        prefix = hist[:, -(n - 1):]                       # [R, n-1]
        match = (win[:, :, :-1] == prefix[:, None, :]).all(-1)
        match &= (win[:, :, -1] >= 0) & (prefix >= 0).all(-1, keepdim=True)
        nxt = win[:, :, -1].clamp(min=0)
        banned = torch.zeros(logits.shape, dtype=torch.int32, device=logits.device)
        banned.scatter_add_(1, nxt, match.to(torch.int32))
        logits = logits.masked_fill(banned > 0, float("-inf"))
    if boundary_rows is not None and bool(boundary_rows.any()):
        logits = logits.masked_fill(boundary_rows[:, None] & WORD_CONT_MASK[None, :], float("-inf"))
    dead = torch.isinf(logits).all(-1)                    # safety: never leave a row with no options
    if bool(dead.any()):
        logits[dead] = base[dead]
    return logits

def sample_from(logits, dc):
    if dc.strategy == "greedy":
        return logits.argmax(-1)
    logits = logits / max(dc.temperature, 1e-5)
    if dc.top_k > 0:
        kth = torch.topk(logits, dc.top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if dc.top_p < 1.0:
        srt, idx = torch.sort(logits, descending=True, dim=-1)
        cum = srt.softmax(-1).cumsum(-1)
        remove = cum > dc.top_p
        remove[:, 1:] = remove[:, :-1].clone(); remove[:, 0] = False
        srt = srt.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, idx, srt)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(-1)

def _decoder_step(model, dec_ids, t, caches, cross, enc_mask, amp):
    R = dec_ids.size(0)
    pos = torch.full((R, 1), t, dtype=torch.long, device=dec_ids.device)
    with amp_ctx(amp):
        h, caches = model.decoder(dec_ids, enc_mask=enc_mask, position_ids=pos, kv_caches=caches, cross_kvs=cross)
        logits = model.lm_head(h[:, -1, :])
    return logits.float(), caches

@torch.no_grad()
def generate_chunk(model, enc_ids, enc_mask, max_new, dc, hist, boundary_rows=None,
                   return_logits=False, amp=True):
    # Greedy / sampling. Returns new tokens [R, max_new] (and raw per-step logits if requested).
    model.eval()
    R, V = enc_ids.size(0), model.cfg.vocab_size
    with amp_ctx(amp):
        enc_out = model.encode(enc_ids, enc_mask)
        cross = model.decoder.precompute_cross_kv(enc_out)
    seen = build_seen(hist, V)
    dec = torch.full((R, 1), BOS_ID, dtype=torch.long, device=enc_ids.device)
    caches, out, raw = None, [], []
    rows = torch.arange(R, device=enc_ids.device)
    for t in range(max_new):
        logits, caches = _decoder_step(model, dec, t, caches, cross, enc_mask, amp)
        if return_logits:
            raw.append(logits.clone())
        logits = apply_penalties(logits, hist, seen, dc, boundary_rows if t == 0 else None)
        nxt = sample_from(logits, dc)
        out.append(nxt)
        seen[rows, nxt] = True
        hist = torch.cat([hist, nxt[:, None]], 1)[:, -dc.history_tokens:]
        dec = nxt[:, None]
    toks = torch.stack(out, 1)
    return (toks, torch.stack(raw, 1)) if return_logits else toks

@torch.no_grad()
def generate_chunk_beam(model, enc_ids, enc_mask, max_new, dc, hist, boundary_rows=None, amp=True):
    # Beam search over a fixed number of tokens (EOS is banned, so all hypotheses have equal length
    # and no length normalisation is needed). Returns the best beam's tokens [B, max_new].
    model.eval()
    B, K, V = enc_ids.size(0), dc.num_beams, model.cfg.vocab_size
    dv = enc_ids.device
    with amp_ctx(amp):
        enc_out = model.encode(enc_ids, enc_mask).repeat_interleave(K, 0)
        cross = model.decoder.precompute_cross_kv(enc_out)
    mask_k = enc_mask.repeat_interleave(K, 0) if enc_mask is not None else None
    hist = hist.repeat_interleave(K, 0)
    seen = build_seen(hist, V)
    br = boundary_rows.repeat_interleave(K, 0) if boundary_rows is not None else None
    scores = torch.full((B, K), -1e9, device=dv); scores[:, 0] = 0.0   # start from one live beam
    base = (torch.arange(B, device=dv) * K)[:, None]
    rows = torch.arange(B * K, device=dv)
    dec = torch.full((B * K, 1), BOS_ID, dtype=torch.long, device=dv)
    toks = torch.empty((B * K, 0), dtype=torch.long, device=dv)
    caches = None
    for t in range(max_new):
        logits, caches = _decoder_step(model, dec, t, caches, cross, mask_k, amp)
        logits = apply_penalties(logits, hist, seen, dc, br if t == 0 else None)
        cand = (scores.view(-1, 1) + F.log_softmax(logits, -1)).view(B, K * V)
        top_s, top_i = cand.topk(K, dim=-1)
        sel = (base + top_i // V).view(-1)                # which parent beam each survivor came from
        tok = (top_i % V).view(-1)
        caches = [(k.index_select(0, sel), v.index_select(0, sel)) for k, v in caches]
        hist, seen, toks = hist.index_select(0, sel), seen.index_select(0, sel), toks.index_select(0, sel)
        toks = torch.cat([toks, tok[:, None]], 1)
        seen[rows, tok] = True
        hist = torch.cat([hist, tok[:, None]], 1)[:, -dc.history_tokens:]
        scores = top_s
        dec = tok[:, None]
    best = (base.view(-1) + scores.argmax(-1))
    return toks.index_select(0, best)

def truncate_words(text, n=128):
    return " ".join(text.split()[:n])

@torch.no_grad()
def generate_continuations(model, prompts, dc, target_words=128, max_chunks=4, batch_size=16,
                           chunk_tokens=None, amp=True, verbose=True):
    # Adaptive chunked generation: each chunk re-encodes the last encoder_len tokens of (prompt + generated)
    # and decodes up to decoder_len new tokens from BOS (exactly the training setup). Rows stop once they
    # have > target_words words (so the target_words-th word is complete).
    model.eval()
    E = model.cfg.encoder_len
    C = chunk_tokens or model.cfg.decoder_len
    assert C <= model.cfg.decoder_len, "never decode past trained decoder positions"
    ids = list(prompts.keys())
    ptok = [tokenizer.encode(prompts[k], allow_special=False) for k in ids]
    gen = [[] for _ in ids]
    done = [False] * len(ids)
    t0 = time.time()
    for c in range(max_chunks):
        active = [i for i in range(len(ids)) if not done[i]]
        if not active:
            break
        active.sort(key=lambda i: min(E, len(ptok[i]) + len(gen[i])))
        for s in range(0, len(active), batch_size):
            batch = active[s:s + batch_size]
            enc, msk = right_pad_batch([(ptok[i] + gen[i])[-E:] for i in batch])
            hist = left_pad([gen[i][-dc.history_tokens:] for i in batch])
            br = (torch.tensor([len(gen[i]) == 0 for i in batch], device=device)
                  if dc.force_word_boundary else None)
            if dc.strategy == "beam":
                toks = generate_chunk_beam(model, enc, msk, C, dc, hist, br, amp=amp)
            else:
                toks = generate_chunk(model, enc, msk, C, dc, hist, br, amp=amp)
            for i, row in zip(batch, toks.tolist()):
                gen[i].extend(row)
        for i in active:
            if len(tokenizer.decode(gen[i]).split()) > target_words:
                done[i] = True
        if verbose:
            print(f"  chunk {c+1}: {sum(done)}/{len(ids)} complete ({time.time()-t0:.0f}s)")
    # U+FFFD only appears if the model emits an invalid byte sequence; drop it rather than submit it.
    return {k: truncate_words(tokenizer.decode(gen[j]).replace("\ufffd", ""), target_words)
            for j, k in enumerate(ids)}

## Submission writing and validation

In [7]:
# ===================== Cell 7: test prompts, submission, validator =====================
def load_test_prompts():
    # Official Phase-1 test prompts: {id: text}. Raw JSON first, the datasets library as fallback.
    try:
        fs = HfFileSystem()
        with fs.open(f"datasets/{HF_REPO}/{TEST_PATH}", "r") as f:
            raw = json.load(f)
        out = {str(k): (v.get("text", "") if isinstance(v, dict) else str(v)) for k, v in raw.items()}
        if out:
            return out
    except Exception as e:
        print("raw test JSON load failed, falling back to datasets:", e)
    from datasets import load_dataset
    ds = load_dataset(HF_REPO, "phase_1", split="test", streaming=True, token=True)
    out = {}
    for row in ds:
        for k, v in row.items():
            if isinstance(v, dict):
                out[str(k)] = v.get("text", "")
            elif isinstance(v, str):
                out[str(k)] = v
    return out

def prompt_length_report(prompts):
    toks = np.array([len(tokenizer.encode(t, allow_special=False)) for t in prompts.values()])
    words = np.array([len(t.split()) for t in prompts.values()])
    print(f"{len(prompts)} prompts | words: mean {words.mean():.0f}, min {words.min()}, max {words.max()}")
    print(f"tokens: mean {toks.mean():.0f} | p50 {np.percentile(toks,50):.0f} | p90 {np.percentile(toks,90):.0f} "
          f"| p95 {np.percentile(toks,95):.0f} | max {toks.max()} | tokens/word {toks.sum()/words.sum():.3f}")
    return toks

def write_submission(preds, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pj = out_dir / "prediction.json"
    pj.write_text(json.dumps({k: {"text": v} for k, v in preds.items()}, ensure_ascii=False))
    zp = out_dir / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    print("wrote", pj, "and", zp)
    return pj, zp

def validate_submission(out_dir, prompts, target_words=128):
    out_dir = Path(out_dir)
    sub = json.loads((out_dir / "prediction.json").read_text())
    problems = []
    if set(sub) != set(prompts):
        problems.append(f"ID mismatch: missing {len(set(prompts)-set(sub))}, extra {len(set(sub)-set(prompts))}")
    wc = {k: len(v.get("text", "").split()) for k, v in sub.items()}
    short = [k for k, n in wc.items() if n < target_words]
    longer = [k for k, n in wc.items() if n > target_words]
    empty = [k for k, n in wc.items() if n == 0]
    bad = [k for k, v in sub.items() if "\ufffd" in v.get("text", "")]
    leak = []
    for k, v in sub.items():
        pw = prompts.get(k, "").split()
        if len(pw) >= 12 and " ".join(pw[:12]) in v.get("text", ""):
            leak.append(k)
    if short: problems.append(f"{len(short)} texts shorter than {target_words} words (e.g. {short[:3]})")
    if longer: problems.append(f"{len(longer)} texts longer than {target_words} words")
    if empty: problems.append(f"{len(empty)} empty texts")
    if bad: problems.append(f"{len(bad)} texts contain U+FFFD (broken bytes)")
    if leak: problems.append(f"{len(leak)} texts appear to contain the prompt")
    with zipfile.ZipFile(out_dir / "submission.zip") as z:
        if z.namelist() != ["prediction.json"]:
            problems.append(f"zip contents {z.namelist()} (must be exactly ['prediction.json'] at root)")
    vals = list(wc.values())
    print(f"{len(sub)} predictions | words min/mean/max = {min(vals)}/{np.mean(vals):.1f}/{max(vals)}")
    if problems:
        print("VALIDATION FAILED:"); [print("  -", p) for p in problems]
    else:
        print("VALIDATION OK")
    return not problems

## Pseudo-test set and local metrics
Built from held-out documents in the same shape as the test set: a 512-word prompt and the next 128 words as the reference. ROUGE-L and BLEU-4 are implemented here, along with repetition statistics. Use the course's official local evaluator for final numbers. BERTScore needs a pretrained RoBERTa, so it is off by default; ask the TAs whether using it for local evaluation only is acceptable.

In [8]:
# ===================== Cell 8: pseudo-test + metrics + decoding sweep =====================
from collections import Counter

def build_pseudo_test(n_max=200, prompt_words=512, cont_words=128):
    docs = [json.loads(l) for l in open(VAL_TEXTS_PATH)]
    by_src = {}
    for d in docs:
        by_src.setdefault(d["source"], []).append(d)
    order, i = [], 0
    while len(order) < n_max and any(i < len(v) for v in by_src.values()):   # round-robin over sources
        for v in by_src.values():
            if i < len(v):
                order.append(v[i])
        i += 1
    prompts, refs, srcs = {}, {}, {}
    for d in order[:n_max]:
        w = d["text"].split()
        if len(w) < prompt_words + cont_words:
            continue
        r = random.Random(d["id"])
        s = r.randint(0, len(w) - prompt_words - cont_words)
        pid = f"pt_{len(prompts)}"
        prompts[pid] = " ".join(w[s:s + prompt_words])
        refs[pid] = " ".join(w[s + prompt_words:s + prompt_words + cont_words])
        srcs[pid] = d["source"]
    print(f"pseudo-test: {len(prompts)} items", Counter(srcs.values()))
    return prompts, refs, srcs

def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rouge_l_f1(hyp, ref):
    h, r = hyp.lower().split(), ref.lower().split()
    if not h or not r:
        return 0.0
    l = _lcs(h, r)
    if l == 0:
        return 0.0
    p, rc = l / len(h), l / len(r)
    return 2 * p * rc / (p + rc)

def corpus_bleu4(hyps, refs):
    num, den, hl, rl = [0] * 4, [0] * 4, 0, 0
    for hyp, ref in zip(hyps, refs):
        h, r = hyp.lower().split(), ref.lower().split()
        hl += len(h); rl += len(r)
        for n in range(1, 5):
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1))
            rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            num[n - 1] += sum(min(c, rc[g]) for g, c in hc.items())
            den[n - 1] += max(0, len(h) - n + 1)
    if hl == 0:
        return 0.0
    logp = sum(math.log((num[i] + 1e-9) / max(den[i], 1)) for i in range(4)) / 4
    bp = 1.0 if hl > rl else math.exp(1 - rl / max(hl, 1))
    return bp * math.exp(logp)

def repetition_stats(texts):
    d2, rep4 = [], []
    for t in texts:
        w = t.lower().split()
        bi = [tuple(w[i:i + 2]) for i in range(len(w) - 1)]
        fg = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
        d2.append(len(set(bi)) / max(1, len(bi)))
        rep4.append(1 - len(set(fg)) / max(1, len(fg)))
    return float(np.mean(d2)), float(np.mean(rep4))

USE_BERTSCORE = False   # see note above

def score_predictions(preds, refs):
    keys = list(refs)
    hyps, rfs = [preds[k] for k in keys], [refs[k] for k in keys]
    out = {"rougeL_f1": float(np.mean([rouge_l_f1(h, r) for h, r in zip(hyps, rfs)])),
           "bleu4": corpus_bleu4(hyps, rfs)}
    out["distinct2"], out["rep4"] = repetition_stats(hyps)
    out["mean_words"] = float(np.mean([len(h.split()) for h in hyps]))
    if USE_BERTSCORE:
        from bert_score import score as bscore
        _, _, f1 = bscore(hyps, rfs, lang="en", verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out

def decoding_sweep(model, prompts, refs, configs, batch_size=16, save_to=None):
    results = []
    for dc in configs:
        t0 = time.time()
        preds = generate_continuations(model, prompts, dc, batch_size=batch_size, verbose=False)
        sc = score_predictions(preds, refs)
        sc.update({"decode": dc.label(), "seconds": round(time.time() - t0, 1)})
        results.append(sc)
        print(f"{dc.label():40s} " + " | ".join(f"{k} {v:.4f}" for k, v in sc.items()
                                                if isinstance(v, float) and k != "seconds")
              + f" | {sc['seconds']}s")
    if save_to:
        Path(save_to).write_text(json.dumps(results, indent=2))
    return results

## Self-tests (run before training; under a minute)
They use a tiny random model in fp32. If any assertion fails, don't train or submit until it's fixed.

In [9]:
# ===================== Cell 9: self-tests =====================
def run_self_tests():
    # TF32 would add ~1e-3 relative noise and mask real bugs, so the tests run in true fp32.
    _tf32 = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    try:
        _run_self_tests()
    finally:
        torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = _tf32

def _run_self_tests():
    torch.manual_seed(0)
    tiny = dict(vocab_size=VOCAB_SIZE, n_encoder_layers=2, n_decoder_layers=2, d_model=128, n_head=4,
                n_kv_head=2, head_dim=32, encoder_len=64, decoder_len=32, max_seq_len=128)
    m = Seq2SeqLM(ModelConfig(**tiny)).to(device).eval()
    init_weights(m, m.cfg)
    with torch.no_grad():                     # make logits non-trivial so comparisons are meaningful
        m.tok_emb.weight.mul_(20)
    plain = DecodeConfig(force_word_boundary=False)

    # 1) incremental (KV cache + cached cross K/V) == full teacher-forced forward
    enc = torch.randint(FIRST_VALID_ID, VOCAB_SIZE, (2, 48), device=device)
    hist0 = left_pad([[], []])
    toks, inc = generate_chunk(m, enc, None, 24, plain, hist0, return_logits=True, amp=False)
    dec_in = torch.cat([torch.full((2, 1), BOS_ID, device=device), toks[:, :-1]], 1)
    with torch.no_grad():
        full = m(enc, dec_in).float()
    rel = ((inc - full).abs().max() / full.abs().max()).item()
    print(f"[1] KV-cache equivalence: rel max diff {rel:.2e}")
    assert rel < 1e-3, rel

    # 2) beam search with 1 beam == greedy
    b1 = generate_chunk_beam(m, enc, None, 24, DecodeConfig(strategy="beam", num_beams=1,
                                                            force_word_boundary=False), hist0, amp=False)
    assert torch.equal(b1, toks), "beam(1) != greedy"
    b4 = generate_chunk_beam(m, enc, None, 12, DecodeConfig(strategy="beam", num_beams=4,
                                                            force_word_boundary=False), hist0, amp=False)
    assert b4.shape == (2, 12)
    print("[2] beam(1)==greedy OK, beam(4) runs")

    # 3) right-padding invariance (encoder states and generated tokens)
    a = torch.randint(FIRST_VALID_ID, VOCAB_SIZE, (40,)).tolist()
    b = torch.randint(FIRST_VALID_ID, VOCAB_SIZE, (64,)).tolist()
    ids, msk = right_pad_batch([a, b])
    with torch.no_grad():
        h_alone = m.encode(torch.tensor([a], device=device))
        h_batch = m.encode(ids, msk)
    d = (h_alone[0] - h_batch[0, :40]).abs().max().item()
    t_alone = generate_chunk(m, torch.tensor([a], device=device), None, 16, plain, left_pad([[]]), amp=False)
    t_batch = generate_chunk(m, ids, msk, 16, plain, left_pad([[], []]), amp=False)
    print(f"[3] padding invariance: encoder diff {d:.2e}, tokens equal {torch.equal(t_alone[0], t_batch[0])}")
    assert d < 1e-3 and torch.equal(t_alone[0], t_batch[0])

    # 4) bidirectional encoder sees the future; a causal one does not
    x = torch.randint(FIRST_VALID_ID, VOCAB_SIZE, (1, 32), device=device)
    y = x.clone(); y[0, -1] = FIRST_VALID_ID + (int(y[0, -1]) - FIRST_VALID_ID + 1) % (VOCAB_SIZE - FIRST_VALID_ID)
    with torch.no_grad():
        dbi = (m.encode(x)[0, 0] - m.encode(y)[0, 0]).abs().max().item()
        mc = Seq2SeqLM(ModelConfig(**tiny, encoder_causal=True)).to(device).eval()
        dca = (mc.encode(x)[0, 0] - mc.encode(y)[0, 0]).abs().max().item()
    print(f"[4] bidirectional: pos0 change {dbi:.2e} (>0) | causal: {dca:.2e} (==0)")
    assert dbi > 0 and dca < 1e-6

    # 5) n-gram ban and word-boundary mask
    t0_, t1_, t2_ = FIRST_VALID_ID + 500, FIRST_VALID_ID + 501, FIRST_VALID_ID + 502
    h = torch.tensor([[t0_, t1_, t2_, t0_, t1_]], device=device)
    lg = apply_penalties(torch.zeros(1, VOCAB_SIZE, device=device), h, build_seen(h, VOCAB_SIZE),
                         DecodeConfig(no_repeat_ngram=3))
    assert torch.isinf(lg[0, t2_]) and not torch.isinf(lg[0, t0_])
    lg2 = apply_penalties(torch.zeros(1, VOCAB_SIZE, device=device), left_pad([[]]),
                          build_seen(left_pad([[]]), VOCAB_SIZE), plain,
                          boundary_rows=torch.tensor([True], device=device))
    assert torch.isinf(lg2[0][WORD_CONT_MASK]).all()
    print("[5] n-gram ban + word-boundary mask OK")

    # 6) checkpoint guard refuses configs without encoder_causal
    cfgd = asdict(m.cfg); cfgd.pop("encoder_causal")
    try:
        cfg_from_dict(cfgd); raise AssertionError("guard did not fire")
    except ValueError:
        print("[6] checkpoint guard OK")

    # 7) MLM corruption sanity
    rng = np.random.default_rng(0)
    arr = np.random.default_rng(1).integers(FIRST_VALID_ID, VOCAB_SIZE, size=(8, 512))
    ids2, lab = mlm_corrupt(arr, rng)
    frac = (lab != -100).mean(); mfrac = (ids2[lab != -100] == MASK_ID).mean()
    print(f"[7] MLM: {frac:.3f} of tokens selected (~0.15), {mfrac:.2f} of them -> <mask> (~0.8)")
    assert 0.12 < frac < 0.18 and 0.7 < mfrac < 0.9
    del m, mc
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("ALL SELF-TESTS PASSED")

run_self_tests()

[1] KV-cache equivalence: rel max diff 4.86e-07
[2] beam(1)==greedy OK, beam(4) runs
[3] padding invariance: encoder diff 4.77e-07, tokens equal True
[4] bidirectional: pos0 change 8.46e-03 (>0) | causal: 0.00e+00 (==0)
[5] n-gram ban + word-boundary mask OK
[6] checkpoint guard OK
[7] MLM: 0.154 of tokens selected (~0.15), 0.83 of them -> <mask> (~0.8)
ALL SELF-TESTS PASSED


## Main: readers, lengths, config, training
Set the run flags below. Steps 1–2 are cheap. Step 3 is the long training run.

In [10]:
# ===================== Cell 10: readers + test prompt lengths =====================
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]
readers = {s: ShardReader(SHARD_DIR / s) for s in SOURCES if (SHARD_DIR / s / "meta.json").exists()}
print("tokens per source:", {k: f"{v.total_tokens/1e6:.1f}M" for k, v in readers.items()})

MIX_OVERRIDE = None   # e.g. {"acts": 0.10, "high_courts": 0.45, "supreme_court": 0.25, "tribunals": 0.20}
_tot = sum(r.total_tokens for r in readers.values())
mix_weights = MIX_OVERRIDE or {k: r.total_tokens / _tot for k, r in readers.items()}
print("mix weights:", {k: round(v, 3) for k, v in mix_weights.items()})

test_prompts = load_test_prompts()
_tok_lens = prompt_length_report(test_prompts)

tokens per source: {'acts': '20.0M', 'high_courts': '65.0M', 'supreme_court': '25.0M', 'tribunals': '40.0M'}
mix weights: {'acts': 0.133, 'high_courts': 0.433, 'supreme_court': 0.167, 'tribunals': 0.267}
raw test JSON load failed, falling back to datasets: datasets/Exploration-Lab/CS-781-Capstone/Phase_1_Pretraining/Test/Phase1- Test.json


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

1000 prompts | words: mean 512, min 512, max 512
tokens: mean 641 | p50 631 | p90 702 | p95 733 | max 1032 | tokens/word 1.252


In [11]:
# ===================== Cell 11: config + training =====================
RUN_NAME = "encdec_bidir_v2"           # unique per run -> its own folder; never reuse a folder across variants
RUN_TRAIN = True

# encoder_len: the p95 test-prompt length (rounded up to 64) covers ~all of the 512-word prompt.
# Larger values cost roughly quadratic encoder attention; 768 is a good T4 compromise.
ENCODER_LEN = int(min(1024, max(512, 64 * math.ceil(np.percentile(_tok_lens, 95) / 64))))
ENCODER_LEN = min(ENCODER_LEN, 768)
DECODER_LEN = 256                      # 128 words ~ 182 tokens -> a single decoding chunk usually suffices

cfg = ModelConfig(vocab_size=VOCAB_SIZE, n_encoder_layers=6, n_decoder_layers=6, d_model=512,
                  n_head=8, n_kv_head=8, head_dim=64, encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN,
                  max_seq_len=1024, encoder_causal=False, mlm_head=True)
tcfg = TrainConfig(
    micro_batch=8, grad_accum=8,       # 64 sequences/step; if OOM: micro_batch=4, grad_accum=16
    peak_lr=6e-4, min_lr=6e-5, warmup_steps=300,
    total_steps=None, plan_hours=7.0, session_hours=8.5,
    mlm_every=4, mlm_prob=0.15, mlm_weight=1.0, boundary_guard=64,
    eval_every=250, log_every=25, ckpt_every_minutes=20,
    output_dir=str(WORK_DIR / "runs" / RUN_NAME),
    resume=True, resume_from=None,     # e.g. "/kaggle/input/<prev-session-dataset>/last.pt"
)
# tcfg = TrainConfig(
#     micro_batch=8, grad_accum=8,       # 64 sequences/step; if OOM: micro_batch=4, grad_accum=16
#     peak_lr=6e-4, min_lr=6e-5, warmup_steps=10,
#     total_steps=None, plan_hours=0.1, session_hours=0.2,
#     mlm_every=4, mlm_prob=0.15, mlm_weight=1.0, boundary_guard=64,
#     eval_every=10, log_every=5, ckpt_every_minutes=2,
#     output_dir=str(WORK_DIR / "runs" / RUN_NAME),
#     resume=True, resume_from=None,     # e.g. "/kaggle/input/<prev-session-dataset>/last.pt"
# )
print(cfg); print(tcfg)

train_loader = PretrainLoader(readers, mix_weights, cfg.encoder_len, cfg.decoder_len, tcfg.micro_batch,
                              seed=tcfg.seed, boundary_guard=tcfg.boundary_guard, mlm_prob=tcfg.mlm_prob)
val_s2s = FixedValSeq2Seq(readers, cfg.encoder_len, cfg.decoder_len, tcfg.val_windows_per_source,
                          guard=tcfg.boundary_guard)
val_mlm = FixedValMLM(readers, cfg.encoder_len, windows_per_source=32, mlm_prob=tcfg.mlm_prob)

_ = [train_loader.get_s2s_batch(i) for i in range(25)]
print(f"loader check: boundary fallback rate {train_loader.fallback_rate():.1%} "
      f"(high values mean many short docs; consider lowering boundary_guard)")
train_loader.stats = {"ok": 0, "fallback": 0}

if RUN_TRAIN:
    model, run_dir, finished = train_seq2seq(cfg, tcfg, train_loader, val_s2s, val_mlm)
    print("finished:", finished, "| run dir:", run_dir)

ModelConfig(vocab_size=32000, n_encoder_layers=6, n_decoder_layers=6, d_model=512, n_head=8, n_kv_head=8, head_dim=64, ff_mult=2.6666666666666665, d_ff=1408, max_seq_len=1024, encoder_len=768, decoder_len=256, norm_type='rmsnorm', norm_eps=1e-05, pos_type='rope', rope_theta=10000.0, mlp_type='swiglu', bias=False, attn_dropout=0.0, resid_dropout=0.0, embd_dropout=0.0, tie_embeddings=True, init_std=0.02, use_sdpa=True, encoder_causal=False, mlm_head=True, arch_name='encdec_bidir_mlm_v2')
TrainConfig(micro_batch=8, grad_accum=8, peak_lr=0.0006, min_lr=6e-05, warmup_steps=300, total_steps=None, plan_hours=7.0, session_hours=8.5, bench_steps=20, weight_decay=0.1, betas=(0.9, 0.95), eps=1e-08, grad_clip=1.0, mlm_every=4, mlm_prob=0.15, mlm_weight=1.0, boundary_guard=64, eval_every=250, val_windows_per_source=64, log_every=25, ckpt_every_minutes=20, seed=1337, output_dir='/kaggle/working/runs/encdec_bidir_v2', resume=True, resume_from=None, fit_schedule_in_session=True)
FixedValSeq2Seq: 256 w

## Decoding sweep on the pseudo-test set, then the final submission

In [12]:
# ===================== Cell 12: decoding sweep (pseudo-test) =====================
RUN_SWEEP = True
CKPT_PATH = WORK_DIR / "runs" / RUN_NAME / "best.pt"
# Inference-only session: CKPT_PATH = Path("/kaggle/input/<dataset>/best.pt")

if RUN_SWEEP:
    model, cfg_loaded, _ = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)
    pt_prompts, pt_refs, pt_srcs = build_pseudo_test(n_max=200)
    sweep = [
        DecodeConfig(strategy="greedy", repetition_penalty=1.1, no_repeat_ngram=4),
        DecodeConfig(strategy="greedy", repetition_penalty=1.0, no_repeat_ngram=3),
        DecodeConfig(strategy="beam", num_beams=4, repetition_penalty=1.0, no_repeat_ngram=4),
        DecodeConfig(strategy="sample", temperature=0.7, top_p=0.9, repetition_penalty=1.1, no_repeat_ngram=4),
        DecodeConfig(strategy="sample", temperature=0.9, top_k=50, top_p=0.95, repetition_penalty=1.05),
    ]
    sweep_results = decoding_sweep(model, pt_prompts, pt_refs, sweep,
                                   save_to=Path(CKPT_PATH).parent / "decoding_sweep.json")

Loaded /kaggle/working/runs/encdec_bidir_v2/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
pseudo-test: 113 items Counter({'high_courts': 57, 'tribunals': 30, 'supreme_court': 17, 'acts': 9})
greedy_rp1.1_ng4                         rougeL_f1 0.2974 | bleu4 0.2023 | distinct2 0.8567 | rep4 0.0069 | mean_words 128.0000 | 30.5s
greedy_rp1.0_ng3                         rougeL_f1 0.2828 | bleu4 0.1762 | distinct2 0.8650 | rep4 0.0033 | mean_words 128.0000 | 26.9s
beam4_rp1.0_ng4                          rougeL_f1 0.3166 | bleu4 0.2296 | distinct2 0.8338 | rep4 0.0113 | mean_words 128.0000 | 35.1s
sample_T0.7_k0_p0.9_rp1.1_ng4            rougeL_f1 0.2727 | bleu4 0.1708 | distinct2 0.9114 | rep4 0.0030 | mean_words 128.0000 | 32.5s
sample_T0.9_k50_p0.95_rp1.05_ng0         rougeL_f1 0.2940 | bleu4 0.2014 | distinct2 0.8958 | rep4 0.0195 | mean_words 128.0000 | 32.8s


In [13]:
# ===================== Cell 13: final submission =====================
RUN_SUBMISSION = True
FINAL_DECODE = DecodeConfig(strategy="greedy", repetition_penalty=1.1, no_repeat_ngram=4)  # <- set from the sweep

if RUN_SUBMISSION:
    # Always submit from best.pt (after training, `model` holds the last step, not the best checkpoint).
    model, _, _ = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)
    torch.manual_seed(SEED)              # reproducible if a sampling strategy is chosen
    preds = generate_continuations(model, test_prompts, FINAL_DECODE, target_words=128,
                                   max_chunks=4, batch_size=16)
    sub_dir = WORK_DIR / "submissions" / f"{RUN_NAME}__{FINAL_DECODE.label()}"
    write_submission(preds, sub_dir)
    ok = validate_submission(sub_dir, test_prompts)
    print(json.dumps({k: preds[k] for k in list(preds)[:2]}, indent=2, ensure_ascii=False)[:1500])

Loaded /kaggle/working/runs/encdec_bidir_v2/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
  chunk 1: 996/1000 complete (219s)
  chunk 2: 1000/1000 complete (222s)
wrote /kaggle/working/submissions/encdec_bidir_v2__greedy_rp1.1_ng4/prediction.json and /kaggle/working/submissions/encdec_bidir_v2__greedy_rp1.1_ng4/submission.zip
1000 predictions | words min/mean/max = 128/128.0/128
VALIDATION OK
{
  "1": "not a case where the Court has declined to entertain the writ petition. The Court has held that even if the petitioner had approached the Court within reasonable time, it would be a futile exercise and would not be treated as a bar for entertaining the writ petition.\" (Emphasis added) 6 11. Similar view has been reiterated in [State of U.P. & Ors. Vs. L. Krishnan and Ors.], AIR 1997 SC 2270; State of Maharashtra Vs. Lisari Devi & Anr., AIR 1984 SC 163; State of Rajasthan Vs. Lipok AO & Ors., AIR 1 995 SC 12